In [1]:
import pandas as pd
import numpy as np
import scipy
import statsmodels.api as sm
from statsmodels.formula.api import ols

In [2]:
df = pd.read_excel('Tensile strength of paper.xlsx')
df

,hardwood concentration 5%,hardwood concentration 10%,hardwood concentration 15%,hardwood concentration 20%
0,7,12,14,19
1,8,17,18,25
2,15,13,19,22
3,11,18,17,23
4,9,19,16,18
5,10,15,18,20


In [3]:
df.columns = [
    'concentration5',
    'concentration10',
    'concentration15',
    'concentration20'
]

print(df.columns)

Index(['concentration5', 'concentration10', 'concentration15',
       'concentration20'],
      dtype='object')


In [4]:
data_r1 = pd.melt(
    df.reset_index(),
    id_vars=['index'],
    value_vars=['concentration5', 'concentration10', 'concentration15', 'concentration20']
)

data_r1.columns = ['index', 'treatments', 'value']

In [5]:
model = ols('value ~ C(treatments)', data=data_r1).fit()

In [6]:
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                  value   R-squared:                       0.746
Model:                            OLS   Adj. R-squared:                  0.708
Method:                 Least Squares   F-statistic:                     19.61
Date:                Tue, 31 Mar 2026   Prob (F-statistic):           3.59e-06
Time:                        18:51:31   Log-Likelihood:                -54.344
No. Observations:                  24   AIC:                             116.7
Df Residuals:                      20   BIC:                             121.4
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
====================================================================================================
                                       coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------------
Intercept                           15.6667      1.041     15.042      0.000      13.494      17.839
C(treatments)[T.concentration15]     1.3333      1.473      0.905      0.376      -1.739       4.406
C(treatments)[T.concentration20]     5.5000      1.473      3.734      0.001       2.428       8.572
C(treatments)[T.concentration5]     -5.6667      1.473     -3.847      0.001      -8.739      -2.594
==============================================================================
Omnibus:                        0.929   Durbin-Watson:                   2.181
Prob(Omnibus):                  0.628   Jarque-Bera (JB):                0.861
Skew:                           0.248   Prob(JB):                        0.650
Kurtosis:                       2.215   Cond. No.                         4.79
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [7]:
aov_table = sm.stats.anova_lm(model, typ = 1)
aov_table

,df,sum_sq,mean_sq,F,PR(>F)
C(treatments),3.0,382.791667,127.597222,19.605207,0.000004
Residual,20.0,130.166667,6.508333,NaN,NaN


In [8]:
import math
t = -1 * scipy.stats.t.ppf(0.025, 20)
n = 6
MSE = 6.508333
lsd = t * math.sqrt(2 * MSE / n)
lsd

np.float64(3.0724225883252476)

In [9]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.multicomp import MultiComparison
mc = MultiComparison(data_r1['value'], data_r1['treatments'])
mcresult = mc.tukeyhsd(0.05)
mcresult.summary()

group1,group2,meandiff,p-adj,lower,upper,reject
concentration10,concentration15,1.3333,0.8022,-2.7892,5.4559,False
concentration10,concentration20,5.5,0.0066,1.3774,9.6226,True
concentration10,concentration5,-5.6667,0.0051,-9.7892,-1.5441,True
concentration15,concentration20,4.1667,0.047,0.0441,8.2892,True
concentration15,concentration5,-7.0,0.0007,-11.1226,-2.8774,True
concentration20,concentration5,-11.1667,0.0,-15.2892,-7.0441,True
